# Matrix formulation of the network measures

This notebook accesses the transition matrix directly through `flywire_tools` and
re-expresses the three measures from the cartoon figure
(`docs/network_measures_cartoon.svg`) in matrix form:

1. **synapse-weighted path count**
2. **input contribution** (into a target)
3. **output contribution** (out of a source)

The subgraph is the one used elsewhere in this repo: start from the lamina input
channels (`L1`–`L5`, `R7`, `R8`) and walk **downstream** up to 5 hops, or start
from the `LC`/`LPLC` types and walk **upstream** up to 5 hops.

---

## Notation

Let the subgraph have $N$ neurons. Two matrices come out of
`Paths._build_transition_matrix`:

| symbol | call | meaning |
|---|---|---|
| $W$ | `_build_transition_matrix(normalize=False)` | raw synapse counts, $W_{ij}$ = synapses $i \to j$ |
| $P$ | `_build_transition_matrix(normalize=True)` | row-stochastic, $P_{ij} = W_{ij} / \sum_k W_{ik}$ |

Both are oriented **along the direction of the walk**, so for an upstream `Paths`
object the matrix is already transposed for you.

## Setup

`flywire_tools` lives in `src/`, and the CSV databases it expects live there too.
We resolve the repository root so the notebook runs from `docs/` or from the root.

In [ ]:
import os
import sys
from pathlib import Path

import numpy as np
import pandas as pd
import scipy.sparse as sp

# --- locate the repo -------------------------------------------------------
REPO_ROOT = Path.cwd().resolve()
while not (REPO_ROOT / "src" / "flywire_tools").exists() and REPO_ROOT != REPO_ROOT.parent:
    REPO_ROOT = REPO_ROOT.parent

SRC_DIR = REPO_ROOT / "src"
CACHE_DIR = REPO_ROOT / "matrix_cache"
CACHE_DIR.mkdir(exist_ok=True)

sys.path.insert(0, str(SRC_DIR))

# several helpers in connectome.py look for CSVs in the *current* directory
os.chdir(SRC_DIR)

print("repo root :", REPO_ROOT)
print("cwd       :", Path.cwd())

In [ ]:
# importing connectome.py pulls in navis + fafbseg and takes a minute or two
from flywire_tools import connectome as fc

DATABASE_FN = str(SRC_DIR / "connections_no_threshold.csv")

STARTING_TYPES = ["L1", "L2", "L3", "L4", "L5", "R7", "R8"]
MAX_HOPS = 5

## 1. Build the subgraph and pull out the transition matrix

`Connectome.get_paths` does the hop-by-hop expansion and returns a `Paths`
object. `Paths._build_transition_matrix` is the direct accessor for the matrix.

Building is expensive (it reads an 800 MB edge table and queries the FlyWire
annotation API), so we cache $W$, $P$ and `node_info` to `matrix_cache/`.

> Note: `Paths.save` currently contains a stray `breakpoint()`, so we cache the
> matrices ourselves rather than using it.

In [ ]:
def dominant_neuropil(edges_df, root_ids):
    """Assign each cell the neuropil holding most of its synapses.

    `neuropil` is an edge attribute, not a node attribute, so we total each
    cell's synapses over both its input and output edges and take the argmax.
    """
    e = edges_df[["pre", "post", "neuropil", "weight"]]
    out = e.groupby(["pre", "neuropil"]).weight.sum()
    inp = e.groupby(["post", "neuropil"]).weight.sum()
    out.index.names = inp.index.names = ["root_id", "neuropil"]
    total = out.add(inp, fill_value=0).reset_index()
    best = total.loc[total.groupby("root_id").weight.idxmax()].set_index("root_id").neuropil
    return pd.Series(root_ids).map(best).fillna("unknown").values


def resolve_cell_class(node_info, src_dir=None):
    """Best-available class label per cell.

    Prefers the FlyWire annotation columns; falls back to the `family` column of
    visual_neuron_types.csv for anything still unlabelled.
    """
    cls = pd.Series("unknown", index=node_info.index, dtype=object)
    for col in ("cell_class", "class", "super_class", "cell_sub_class"):
        if col in node_info.columns:
            vals = node_info[col].astype(str)
            fill = cls.eq("unknown") & ~vals.isin(["nan", "None", ""])
            cls[fill] = vals[fill]

    vnt_fn = Path(src_dir or SRC_DIR) / "visual_neuron_types.csv"
    if cls.eq("unknown").any() and vnt_fn.exists():
        vnt = pd.read_csv(vnt_fn).drop_duplicates("root_id").set_index("root_id").family
        fallback = node_info.root_id.map(vnt)
        fill = cls.eq("unknown") & fallback.notna()
        cls[fill] = fallback[fill].astype(str)
    return cls.values


def build_or_load(source, max_hops=MAX_HOPS, direction="downstream", tag=None, rebuild=False):
    """Return (W, P, node_info) for a subgraph, caching to matrix_cache/.

    W : csr_matrix, raw synapse counts along the walk direction
    P : csr_matrix, row-stochastic version of W
    node_info : DataFrame aligned to the matrix rows/columns, with added
                `neuropil` and `cell_class` columns used for sorting
    """
    tag = tag or f"{'-'.join(np.atleast_1d(source))}_{direction}_{max_hops}hops_v2"
    w_fn = CACHE_DIR / f"{tag}_W.npz"
    p_fn = CACHE_DIR / f"{tag}_P.npz"
    n_fn = CACHE_DIR / f"{tag}_nodes.csv"

    if not rebuild and w_fn.exists() and p_fn.exists() and n_fn.exists():
        print(f"loading cached matrices ({tag}) from {CACHE_DIR}")
        return sp.load_npz(w_fn), sp.load_npz(p_fn), pd.read_csv(n_fn)

    print(f"cache miss ({tag}); rebuilding with get_paths() -- this reads the full "
          f"edge table and expands {max_hops} hops downstream, which can take hours...")
    conn = fc.Connectome(database_fn=DATABASE_FN)
    conn.set_materialization(783)

    root_ids = []
    for cell_type in np.atleast_1d(source):
        root_ids += conn.lookup_root_ids(cell_type)
    paths = conn.get_paths(root_ids, max_hops=max_hops, direction=direction)

    # --- the direct accessor ---
    W = paths._build_transition_matrix(normalize=False)
    P = paths._build_transition_matrix(normalize=True)

    # node_info rows are already sorted by root_id, matching paths.node_ids
    node_info = paths.node_info.reset_index(drop=True)
    assert np.array_equal(node_info.root_id.values, paths.node_ids)

    node_info["neuropil"] = dominant_neuropil(paths.edges_df, node_info.root_id.values)
    node_info["cell_class"] = resolve_cell_class(node_info)

    sp.save_npz(w_fn, W)
    sp.save_npz(p_fn, P)
    node_info.to_csv(n_fn, index=False)
    return W, P, node_info


W, P, node_info = build_or_load(STARTING_TYPES, direction="downstream")
node_ids = node_info.root_id.values

print(f"N nodes    : {W.shape[0]:,}")
print(f"N edges    : {W.nnz:,}")
print(f"total syn  : {W.sum():,.0f}")
print(f"neuropils  : {node_info.neuropil.nunique()}   classes: {node_info.cell_class.nunique()}")
node_info.head()

In [ ]:
# sanity check: P is row-stochastic wherever the node has outgoing edges
row_sums = np.asarray(P.sum(axis=1)).ravel()
has_out = np.asarray(W.sum(axis=1)).ravel() > 0
print("rows with outgoing edges :", has_out.sum())
print("max |rowsum - 1|         :", np.abs(row_sums[has_out] - 1).max())
print("dead-end rows sum to     :", np.unique(row_sums[~has_out]))

## 2. Absorption

The walk is stopped on first arrival at an absorbing set $A$ (the lamina input
channels for an upstream walk, the LC/LPLC targets for a downstream walk). With
$\mathbf{a}$ the boolean indicator of $A$,

$$P_{\text{eff}} = \operatorname{diag}(\neg\,\mathbf{a}) \, P$$

zeroes the outgoing rows of absorbing nodes. Truncating at $H$ hops, the
accumulated mass is

$$\mathbf{N}_H = \sum_{h=0}^{H} P_{\text{eff}}^{\,h},$$

and the **first-passage probability** from $i$ to absorbing node $j$ is
$\big[\mathbf{N}_H\big]_{ij}$ restricted to $j \in A$. As $H \to \infty$ this is
the usual absorbing-chain fundamental matrix $\mathbf{N} = (I - P_{\text{eff}})^{-1}$.

In [ ]:
def absorbing_mask(node_info, absorbing_types):
    """Boolean mask over matrix rows for nodes whose cell_type matches."""
    types = node_info.cell_type.astype(str).values
    return np.isin(types, np.atleast_1d(absorbing_types))


def effective_matrix(M, absorb_mask):
    """diag(~a) @ M -- zero the outgoing rows of absorbing nodes."""
    keep = sp.diags((~absorb_mask).astype(M.dtype))
    return (keep @ M).tocsr()


# for the downstream walk, absorb at the LC / LPLC targets
types_str = node_info.cell_type.astype(str)
TARGET_TYPES = sorted(types_str[types_str.str.match(r"^(LC|LPLC)\d", na=False)].unique())
print(f"{len(TARGET_TYPES)} target types:", TARGET_TYPES[:12], "...")

absorb = absorbing_mask(node_info, TARGET_TYPES)
P_eff = effective_matrix(P, absorb)
W_eff = effective_matrix(W, absorb)
print("absorbing nodes:", absorb.sum())

## 3. The three measures in matrix form

Let $S$ be the source set and $T$ the target set, and let

$$C \;=\; \sum_{h=1}^{H} W^{h}$$

be the **synapse-weighted path count** matrix: $C_{st}$ counts every directed
walk of length $\le H$ from $s$ to $t$, each weighted by the product of the
synapse counts along it. This is exactly the quantity written out edge-by-edge in
the cartoon,

$$W(s \to t) \;=\; \sum_{\text{paths } \pi:\, s \to t}\; \prod_{(i,j) \in \pi} w_{ij}.$$

The two contributions are the two ways of normalising the same matrix:

$$\text{IC}(s \to t) \;=\; \frac{C_{st}}{\sum_{s' \in S} C_{s't}}
\qquad\text{(normalise down the column)}$$

$$\text{OC}(s \to t) \;=\; \frac{C_{st}}{\sum_{t' \in T} C_{st'}}
\qquad\text{(normalise across the row)}$$

A useful identity: because $P$ is $W$ with its rows normalised,
$\sum_h P^h$ **is** the output-contribution matrix, and running the same
accumulation on the column-stochastic matrix $\big(W^{\mathsf T}\big)$
row-normalised gives the input-contribution matrix. The path count is the
unnormalised version of both.

In [ ]:
def path_contributions(M, src_idx, tgt_idx, max_hops, batch=256, dtype=np.float64):
    """Synapse-weighted path counts from sources to targets, memory-bounded.

    For C = sum_{h=1..max_hops} M^h restricted to the source rows, this returns only the two
    aggregates the contribution measures need -- the target columns C[:, tgt] and the per-
    source total C.sum(1) -- so the full (and, after a few hops, effectively dense)
    reachability is never materialised. Sources are processed in batches of `batch` rows so
    the intermediate frontier stays small.

    Returns (sub, rowsum): `sub` is a dense (len(src_idx), len(tgt_idx)) path-count matrix;
    `rowsum` is a (len(src_idx),) vector of total downstream path counts per source.
    """
    N = M.shape[0]
    src_idx = np.asarray(src_idx)
    tgt_idx = np.asarray(tgt_idx)
    n_src, n_tgt = len(src_idx), len(tgt_idx)
    # column selector (N x n_tgt): picks the target columns via a cheap sparse matmul
    sel = sp.csr_matrix((np.ones(n_tgt, dtype=dtype), (tgt_idx, np.arange(n_tgt))),
                        shape=(N, n_tgt))
    sub = np.zeros((n_src, n_tgt), dtype=dtype)
    rowsum = np.zeros(n_src, dtype=dtype)
    for b0 in range(0, n_src, batch):
        b1 = min(b0 + batch, n_src)
        rows = src_idx[b0:b1]
        X = sp.csr_matrix(
            (np.ones(len(rows), dtype=dtype), (np.arange(len(rows)), rows)),
            shape=(len(rows), N),
        )
        for _ in range(max_hops):
            X = (X @ M).tocsr()
            rowsum[b0:b1] += np.asarray(X.sum(axis=1)).ravel()
            sub[b0:b1] += np.asarray((X @ sel).todense())
    return sub, rowsum


def output_contribution(sub, rowsum):
    """OC = C[:, tgt] / rowsum(C) -- share of the source's output reaching each target."""
    with np.errstate(invalid="ignore", divide="ignore"):
        return np.where(rowsum[:, None] > 0, sub / rowsum[:, None], 0.0)


def input_contribution(sub):
    """IC = C[:, tgt] / colsum over the source set."""
    denom = sub.sum(axis=0)                            # over all sources in S
    with np.errstate(invalid="ignore", divide="ignore"):
        return np.where(denom[None, :] > 0, sub / denom[None, :], 0.0)

In [ ]:
src_idx = np.flatnonzero(node_info.cell_type.astype(str).isin(STARTING_TYPES).values)
tgt_idx = np.flatnonzero(absorb)
print(f"{len(src_idx):,} sources, {len(tgt_idx):,} targets")

# path_contributions is the expensive step (batched sparse matmuls over MAX_HOPS), so cache
# its two outputs -- sub (source x target path counts) and rowsum (per-source totals) -- to
# matrix_cache/. The cache key hashes everything the result depends on: the source/target
# index sets, MAX_HOPS, and the W_eff matrix itself, so it invalidates automatically if any
# of them change. OC / IC are cheap normalisations of sub, so they are always recomputed.
import hashlib


def load_or_compute_contributions(M, src_idx, tgt_idx, max_hops, rebuild=False):
    """Return (sub, rowsum) from path_contributions, caching the result to matrix_cache/."""
    h = hashlib.sha1()
    for arr in (np.asarray(src_idx), np.asarray(tgt_idx),
                M.indptr, M.indices, M.data):
        h.update(np.ascontiguousarray(arr).tobytes())
    h.update(np.int64(max_hops).tobytes())
    fn = CACHE_DIR / f"contributions_{h.hexdigest()[:16]}.npz"

    if not rebuild and fn.exists():
        print(f"loading cached path contributions from {fn.name}")
        d = np.load(fn)
        return d["sub"], d["rowsum"]

    print("cache miss; computing path contributions (batched matmuls over "
          f"{max_hops} hops)...")
    # use W_eff so paths terminate on first arrival at a target. Only the target columns and
    # the per-source totals are kept (in source batches), so the dense reachability never
    # blows up.
    sub, rowsum = path_contributions(M, src_idx, tgt_idx, max_hops)
    np.savez(fn, sub=sub, rowsum=rowsum)
    print(f"saved path contributions to {fn.name}")
    return sub, rowsum


sub, rowsum = load_or_compute_contributions(W_eff, src_idx, tgt_idx, MAX_HOPS)
OC = output_contribution(sub, rowsum)
IC = input_contribution(sub)

print("sub:", sub.shape, f"nonzero {np.count_nonzero(sub) / sub.size:.2%}")
print("OC :", OC.shape, "row sums (should be <= 1):", OC.sum(1)[:5].round(3))
print("IC :", IC.shape, "col sums (should be ~1):", IC.sum(0)[:5].round(3))

## 4. Visualising the transition matrix, averaged by cell type

Collapse the $N \times N$ matrix to $K \times K$ with one row and column per cell
type. With $G \in \{0,1\}^{N \times K}$ the type-membership indicator,

$$\big[G^{\mathsf T} W G\big]_{ab} = \sum_{i \in a} \sum_{j \in b} W_{ij},$$

so the **mean** synapse count between a cell of type $a$ and a cell of type $b$ is
that block sum divided by $n_a n_b$.

In [ ]:
def type_indicator(node_info, type_col="cell_type"):
    """Return (G, types, counts) with G[i, k] = 1 if node i has type k."""
    types_str = node_info[type_col].astype(str).fillna("unknown").values
    types, codes = np.unique(types_str, return_inverse=True)
    counts = np.bincount(codes, minlength=len(types))
    G = sp.csr_matrix(
        (np.ones(len(codes)), (np.arange(len(codes)), codes)),
        shape=(len(codes), len(types)),
    )
    return G, types, counts


def aggregate_by_type(M, node_info, how="mean"):
    """Collapse an N x N matrix to K x K, one row/col per cell type."""
    G, types, counts = type_indicator(node_info)
    block = np.asarray((G.T @ M @ G).todense())
    if how == "mean":
        block = block / np.outer(counts, counts)
    return block, types, counts


W_type, type_names, type_counts = aggregate_by_type(W, node_info, how="mean")

# order the types by mean level so the matrix reads layer-by-layer
level_by_type = node_info.groupby(node_info.cell_type.astype(str))["level"].mean()
order = np.argsort([level_by_type.get(t, np.inf) for t in type_names])
W_type = W_type[np.ix_(order, order)]
type_names = type_names[order]
type_counts = type_counts[order]

print(f"{len(type_names)} cell types")
pd.DataFrame({"type": type_names, "n_cells": type_counts}).head(15)

In [ ]:
import plotly.graph_objects as go


def heatmap(M, labels, title, log=True, colorscale="Greys", height=800, hoverdata=True):
    """Zoomable heatmap. Drag to zoom, double-click to reset.

    `hoverdata=False` drops the per-cell raw-value array from the payload -- for very large
    matrices, sending a second full-size array to the browser is what makes the render hang.
    """
    Z = np.log10(M + 1e-6) if log else M
    trace = dict(z=Z, x=labels, y=labels, colorscale=colorscale,
                 colorbar=dict(title="log10 syn" if log else "syn"))
    if hoverdata:
        trace["customdata"] = M
        trace["hovertemplate"] = "pre %{y}<br>post %{x}<br>%{customdata:.3f} syn<extra></extra>"
    fig = go.Figure(go.Heatmap(**trace))
    fig.update_layout(
        title=title,
        height=height,
        width=height,
        xaxis=dict(title="post-synaptic type", showticklabels=len(labels) <= 80),
        yaxis=dict(title="pre-synaptic type", autorange="reversed",
                   showticklabels=len(labels) <= 80),
    )
    return fig


# The subgraph has thousands of cell types, so the full type x type matrix is far too large
# to serialise to the browser (an 8000x8000 z + customdata is >1 GB of JSON -- this is what
# made the cell hang for hours). Show the most-connected types instead; raise MAX_TYPES to
# see more, or block-downsample like section 5 if you need every type at once.
MAX_TYPES = 200
if len(type_names) > MAX_TYPES:
    strength = np.asarray(W_type).sum(0) + np.asarray(W_type).sum(1)   # total in+out per type
    keep = np.sort(np.argsort(strength)[::-1][:MAX_TYPES])             # keep the level ordering
    W_show, labels_show = W_type[np.ix_(keep, keep)], type_names[keep]
    print(f"{len(type_names):,} cell types -> showing the {MAX_TYPES} most-connected")
else:
    W_show, labels_show = W_type, type_names

heatmap(W_show, labels_show, f"Mean synapses per cell pair, top {len(labels_show)} types")

## 5. Per-cell matrix, grouped by neuropil → cell class → cell type

One row and column per *individual* neuron, restricted to **one optic hemisphere**
(`HEMISPHERE`) and the **main visual / LC-target neuropils** (`MAIN_NEUROPILS`: retina,
lamina, medulla, lobula, lobula plate, AOTU, PVLP, PLP) so the labelling is not swamped by
the long tail of small central-brain neuropils. The R7/R8 photoreceptors get their own
synthetic **retina** neuropil (they otherwise fold into lamina/medulla). Within that subset
we keep only the most-connected cell types (`TOP_TYPES`). Rows are then ordered by a
three-level hierarchy so related cells sit in contiguous blocks:

1. **neuropil** — anatomically ordered along the visual pathway
   (retina → LA → ME → LO → LOP → AOTU → PVLP → PLP), each cell assigned the
   neuropil holding the majority of its synapses (R7/R8 forced to retina);
2. **cell class** within a neuropil, ordered by mean hop level;
3. **cell type** within a class, again by mean level, then by `root_id`.

Solid separators mark neuropil boundaries and faint ones mark class boundaries.
`downsample_max` block-max reduces the matrix for the browser, and `zoom_block` renders any
sub-range at full resolution. A final cell summarises the remaining **non-optic** neuropils
at the neuropil level, where connectivity is dominated by within-neuropil (self) links.

In [ ]:
# ---- restrict to one optic hemisphere and the main visual / LC-target neuropils ----
# This drops the long tail of tiny central-brain neuropils that cluttered the labelling.
HEMISPHERE = "R"                                     # "R" or "L": which optic-lobe side to show
MAIN_NEUROPILS = ["retina", "LA", "ME", "LO", "LOP", "AOTU", "PVLP", "PLP", "targets"]

# --- manual neuropil overrides -------------------------------------------------------------
# Force a few cell groups into anatomically sensible neuropils regardless of what the
# dominant-neuropil assignment from the connectome says:
#   * photoreceptors R1-6, R7, R8  -> synthetic 'retina'   (they otherwise fold into LA / ME)
#   * lamina monopolar cells L1-L5 -> 'LA'                 (so they head the lamina block)
#   * LC / LPLC absorbing targets  -> synthetic 'targets'  (gathered into one block at the end)
# Everything keeps its _R / _L side suffix.
_npil = node_info.neuropil.astype(str)
_side = _npil.str.extract(r"_(R|L)$")[0]                        # side from the dominant neuropil
_side = _side.fillna(node_info.side.astype(str).map({"right": "R", "left": "L"}))
_ct = node_info.cell_type.astype(str)
is_retina = (_ct.str.match(r"^R[1-8]", na=False) | _ct.eq("R1-6")).values
is_lamina = _ct.isin(["L1", "L2", "L3", "L4", "L5"]).values
is_target = _ct.isin(TARGET_TYPES).values                      # the LC / LPLC absorbing targets
npil_label = _npil.copy()
npil_label = npil_label.mask(is_retina, "retina_" + _side.fillna(""))
npil_label = npil_label.mask(is_lamina, "LA_" + _side.fillna(""))
npil_label = npil_label.mask(is_target, "targets_" + _side.fillna(""))

npil_base = npil_label.str.rsplit("_", n=1).str[0]             # strip the _R / _L side suffix
npil_side = npil_label.str.extract(r"_(R|L)$")[0]             # R / L / NaN (unsided neuropils)
main_keep = (npil_base.isin(MAIN_NEUROPILS) & (npil_side == HEMISPHERE)).values
print(f"hemisphere {HEMISPHERE}, neuropils {MAIN_NEUROPILS} -> "
      f"{main_keep.sum():,} / {len(main_keep):,} cells")

# within that subset, keep only the most-connected cell types so the matrix stays legible
TOP_TYPES = 200
deg = np.asarray(W.sum(1)).ravel() + np.asarray(W.sum(0)).ravel()   # per-cell in+out synapses
G_all, all_type_names, _ = type_indicator(node_info)
type_strength = np.asarray(G_all.T @ (deg * main_keep)).ravel()     # strength within the subset
top_types = set(all_type_names[np.argsort(type_strength)[::-1][:TOP_TYPES]])
# always keep the seed input types (L1-L5, R7, R8) and every LC/LPLC target, even if some fall
# outside the top-N by strength (e.g. R8 sits just past the cut).
top_types |= set(STARTING_TYPES) | set(TARGET_TYPES)
cell_keep = main_keep & node_info.cell_type.astype(str).isin(top_types).values
print(f"top {TOP_TYPES} types (+ seeds & targets) within the subset -> {cell_keep.sum():,} cells")

# anatomical ordering along the visual pathway; anything unlisted sorts after, alphabetically.
NEUROPIL_ORDER = ["retina", "LA", "ME", "LO", "LOP", "AOTU", "PVLP", "PLP", "targets"]


def neuropil_rank(name):
    return NEUROPIL_ORDER.index(name) if name in NEUROPIL_ORDER else len(NEUROPIL_ORDER)


info = node_info[cell_keep].reset_index(drop=True)
W_keep = W[cell_keep][:, cell_keep].tocsr()
info["_type"] = info.cell_type.astype(str)
info["_class"] = info.cell_class.astype(str)
info["_npil"] = npil_base[cell_keep].reset_index(drop=True)    # base name (retina/LA/ME/...)
# collapse each cell TYPE to its majority neuropil so every type is a single contiguous block.
# Without this, a few stray cells (e.g. one C2 assigned to LA while the rest are in ME) make the
# type show up as a duplicate block in a second neuropil band.
info["_npil"] = info.groupby("_type")["_npil"].transform(lambda s: s.value_counts().idxmax())

# rank neuropils anatomically so the visual pathway reads top-left to bottom-right. Only the
# neuropil grouping is fixed here; cell order *within* each neuropil is set by the two-level
# dendrogram clustering below, so no cell-type / level pre-sort is needed.
info["_np_rank"] = info._npil.map(neuropil_rank)
sort_key = info.sort_values(["_np_rank", "_npil", "root_id"], kind="stable")
perm = sort_key.index.values
W_sorted = W_keep[perm][:, perm].tocsr()
sorted_info = info.iloc[perm].reset_index(drop=True)

sorted_types = sorted_info._type.values
sorted_class = sorted_info._class.values
sorted_npil = sorted_info._npil.values


# --- two-level dendrogram (hierarchical-clustering) ordering, seaborn.clustermap-style, but
# applied *within each neuropil band* so the arbitrary anatomical grouping is preserved:
#   1) WITHIN each cell type -- order the individual cells by their own connectivity vectors;
#   2) BETWEEN cell types    -- order the type blocks by the type's MEAN connectivity vector.
# To stay memory-safe (the full N x N band densifies to many GB), each cell's feature vector is
# its synapses onto / from every cell TYPE (log-compressed), a compact dense (N x 2K) matrix. ---
from scipy.cluster.hierarchy import leaves_list, linkage, optimal_leaf_ordering
from scipy.spatial.distance import pdist

CLUSTER = True                       # two-level dendrogram leaf ordering (clustermap-style)
CLUSTER_METHOD = "average"           # linkage method (clustermap default)
CLUSTER_METRIC = "euclidean"         # distance metric between feature vectors
ALPHA_TYPE_BANDS = {"LA"}            # bands whose type blocks are ordered alphabetically (L1..L5)


def dendrogram_order(F, method=CLUSTER_METHOD, metric=CLUSTER_METRIC):
    """Optimal-leaf-ordering of the rows of feature matrix `F` via hierarchical clustering.

    `F` has one row per item (a cell, or a cell-type mean) and one column per feature. Returns
    the dendrogram leaves after optimal ordering, so similar rows end up adjacent -- exactly
    the row/column reordering seaborn.clustermap performs.
    """
    n = F.shape[0]
    if n <= 2:
        return np.arange(n)
    D = pdist(np.asarray(F, dtype=float), metric=metric)
    if D.size == 0 or not np.isfinite(D).all() or float(D.max()) == 0.0:
        return np.arange(n)                            # degenerate: nothing to reorder
    Z = optimal_leaf_ordering(linkage(D, method=method), D)
    return leaves_list(Z)


if CLUSTER:
    # per-cell feature vectors: synapses onto (out) and from (in) each cell TYPE, log-
    # compressed since synapse counts are heavy-tailed. Dense but small -- (N x 2K) instead
    # of the (N x N) profile that blew up memory.
    Gt, _, _ = type_indicator(sorted_info, type_col="_type")
    F = np.log1p(np.hstack([np.asarray((W_sorted @ Gt).todense()),
                            np.asarray((W_sorted.T @ Gt).todense())]))

    new_order = []
    # neuropil bands in their current (anatomical) order -- unique labels by first appearance.
    band_names = sorted_npil[np.flatnonzero(np.r_[True, sorted_npil[1:] != sorted_npil[:-1]])]
    for npil in band_names:                                  # bands stay in anatomical order
        band = np.flatnonzero(sorted_npil == npil)
        btypes = sorted_types[band]
        _, first = np.unique(btypes, return_index=True)
        block_types = btypes[np.sort(first)]                 # types in this band (current order)
        cells_by_type = [band[btypes == t] for t in block_types]

        # 1) within each cell type: order individual cells by their own feature vectors
        cells_by_type = [blk[dendrogram_order(F[blk])] for blk in cells_by_type]

        # 2) between cell types: the lamina (LA) types are forced into alphabetical order
        # (L1..L5); every other band orders its type blocks by each type's mean feature vector
        if npil in ALPHA_TYPE_BANDS:
            torder = np.argsort(block_types, kind="stable")
        elif len(block_types) > 2:
            type_means = np.vstack([F[blk].mean(axis=0) for blk in cells_by_type])
            torder = dendrogram_order(type_means)
        else:
            torder = np.arange(len(block_types))
        for i in torder:
            new_order.extend(cells_by_type[i].tolist())

    new_order = np.asarray(new_order)
    W_sorted = W_sorted[new_order][:, new_order].tocsr()
    sorted_info = sorted_info.iloc[new_order].reset_index(drop=True)
    sorted_types = sorted_info._type.values
    sorted_class = sorted_info._class.values
    sorted_npil = sorted_info._npil.values
    print(f"clustered {len(new_order):,} cells within {len(band_names)} neuropil bands "
          f"(cells within type, type blocks by mean vector)")


def blocks(labels):
    """Start index, mid index and name of each run of equal labels."""
    change = np.flatnonzero(np.r_[True, labels[1:] != labels[:-1]])
    edges = np.r_[change, len(labels)]
    return change, (edges[:-1] + edges[1:]) // 2, labels[change]


type_start, type_mid, type_name = blocks(sorted_types)
npil_start, npil_mid, npil_name = blocks(sorted_npil)

print(f"{W_sorted.shape[0]:,} cells")
print(f"  {len(npil_name)} neuropil blocks : {list(npil_name)}")
print(f"  {len(type_name)} type blocks")

In [ ]:
def downsample_max(M, target=2000):
    """Block-max downsample a sparse matrix to about target x target (vectorised).

    Replaces the old ``np.maximum.at`` scatter (a Python-level, O(nnz) unbuffered op that is
    extremely slow) with a sort + ``np.maximum.reduceat`` block-max over the non-zeros.
    """
    N = M.shape[0]
    if N <= target:
        return np.asarray(M.todense()), 1
    factor = int(np.ceil(N / target))
    n_out = int(np.ceil(N / factor))
    coo = M.tocoo()
    if coo.nnz == 0:
        return np.zeros((n_out, n_out)), factor
    lin = (coo.row // factor) * n_out + (coo.col // factor)     # linear block id per nnz
    order = np.argsort(lin, kind="stable")
    lin_s, data_s = lin[order], coo.data[order]
    uniq, starts = np.unique(lin_s, return_index=True)
    # block_max = np.maximum.reduceat(data_s, starts)            # max within each block
    block_max = np.add.reduceat(data_s, starts) / np.diff(np.append(starts, data_s.size))  # mean within each block
    out = np.zeros(n_out * n_out, dtype=float)
    out[uniq] = block_max
    return out.reshape(n_out, n_out), factor


def symlog(x, linthresh=1.0):
    """Symmetric-log transform: linear within +-linthresh, log10-scaled beyond."""
    x = np.asarray(x, dtype=float)
    mag = np.abs(x) / linthresh
    # feed log10 a >=1 argument everywhere (masked to 1 in the linear region) so zeros
    # don't raise a divide-by-zero warning; the outer where then selects the right branch
    return np.where(mag <= 1.0, x / linthresh,
                    np.sign(x) * (1.0 + np.log10(np.where(mag > 1.0, mag, 1.0))))


def symlog_colorbar(vmax, linthresh=1.0, title="synapses"):
    """Colorbar dict with ticks placed in symlog space but labelled in raw units."""
    top_exp = int(np.ceil(np.log10(vmax))) if vmax > linthresh else 0
    orig = [0.0, float(linthresh)] + [10.0 ** e for e in range(1, top_exp + 1)]
    return dict(title=title,
                tickvals=[float(symlog(v, linthresh)) for v in orig],
                ticktext=[f"{int(v):,}" for v in orig])


# colour scale for every heatmap in this section: symlog (linear below LINTHRESH
# synapses, log10-scaled above) or a plain linear scale. Flip USE_SYMLOG to compare.
USE_SYMLOG = True          # set False for a linear colour scale
LINTHRESH = 1.0
VMAX = None                # colour-scale ceiling in raw synapses; None = use the data max


def colour_z(M):
    """(z-values, colorbar, zmax) for `M` under the current colour-scale settings.

    VMAX caps the top of the colour scale (in raw synapses); None uses the data max.
    Values above VMAX still render, saturated at the top colour.
    """
    vmax = max(float(M.max()) if VMAX is None else float(VMAX), LINTHRESH)
    if USE_SYMLOG:
        return (symlog(M, LINTHRESH), symlog_colorbar(vmax, LINTHRESH),
                float(symlog(vmax, LINTHRESH)))
    return M, dict(title="synapses"), vmax


def zoom_block(lo, hi):
    """Full-resolution view of rows/cols [lo, hi) of the sorted matrix."""
    sub = np.asarray(W_sorted[lo:hi, lo:hi].todense())
    labels = [
        f"{n} | {c} | {t} {r}"
        for n, c, t, r in zip(
            sorted_npil[lo:hi], sorted_class[lo:hi], sorted_types[lo:hi],
            sorted_info.root_id[lo:hi],
        )
    ]
    z_sub, cbar, zmax = colour_z(sub)
    fig = go.Figure(go.Heatmap(
        z=z_sub, customdata=sub, zmin=0, zmax=zmax,
        hovertemplate="pre %{y}<br>post %{x}<br>%{customdata:,.0f} syn<extra></extra>",
        colorscale="magma", colorbar=cbar))
    fig.update_layout(title=f"cells [{lo}:{hi}]", height=750, width=750,
                      yaxis=dict(autorange="reversed"))
    return fig


def add_separators(fig, starts, n, color, width, dash=None):
    """Draw grouping separators on both axes of a square heatmap.

    A block that starts at cell ``s`` has its boundary at ``s - 0.5`` (the edge between cell
    s-1 and cell s), not at ``s`` -- which is the *centre* of the first cell of the block.
    Drawing at ``s - 0.5`` puts each divider exactly between rows/columns, so it no longer
    lands on top of the block's (centre-aligned) tick label.
    """
    for s in starts[1:]:
        b = s - 0.5
        for axis in ("x", "y"):
            kw = dict(x0=b, x1=b, y0=-0.5, y1=n - 0.5) if axis == "x" else \
                 dict(x0=-0.5, x1=n - 0.5, y0=b, y1=b)
            fig.add_shape(type="line", line=dict(color=color, width=width, dash=dash),
                          layer="above", **kw)


W_small, factor = downsample_max(W_sorted, target=1200)
n_small = W_small.shape[0]
print(f"downsampled {W_sorted.shape[0]:,} -> {n_small:,} (factor {factor})")

# colour mapping controlled by USE_SYMLOG / VMAX above (symlog vs. linear, capped at VMAX)
z_main, cbar_main, zmax_main = colour_z(W_small)
fig = go.Figure(
    go.Heatmap(
        z=z_main,
        customdata=W_small,
        zmin=0,
        zmax=zmax_main,
        hovertemplate="pre %{y}<br>post %{x}<br>%{customdata:,.0f} syn<extra></extra>",
        colorscale="magma",
        showscale=True,
        colorbar=cbar_main,
    )
)

# white neuropil separators only -- cell-class dividers are dropped (classes omitted)
add_separators(fig, npil_start // factor, n_small, "rgba(255,255,255,0.75)", 0.7)

# --- neuropil names as a real top axis (xaxis2) instead of free-floating annotations. With
# matches="x" it shares the bottom axis' range, so it pans and zooms in lock-step with the
# cell-type ticks. A fully transparent dummy point anchors the overlay axis so it renders. ---
NPIL_DISPLAY = {"retina": "retina"}                    # prettier labels for the figure
npil_tickpos = list(npil_mid // factor)
npil_ticktext = [NPIL_DISPLAY.get(n, n) for n in npil_name]
px_per_col = 880 / n_small                           # figure is 880 px wide

fig.add_trace(go.Scatter(x=[npil_tickpos[0] if npil_tickpos else 0], y=[0],
                         xaxis="x2", yaxis="y", mode="markers",
                         marker=dict(opacity=0), hoverinfo="skip", showlegend=False))

# --- "mip-map" cell-type ticks: show a type label only where there is room, prioritising the
# types with the most cells. Place the largest blocks first and skip any that fall within
# LABEL_GAP_PX of an already-placed label; the survivors go on *both* the bottom (post-
# synaptic) and left (pre-synaptic) spines, so the two label sets are identical. ---
LABEL_GAP_PX = 12                                    # min on-screen spacing between type labels
# ALL_TYPE_LABELS: True -> show *every* cell-type label (great for the interactive plot, where the
# type names become readable as you zoom in); False -> keep only the "mip-map" subset that fits.
ALL_TYPE_LABELS = True
type_size = np.diff(np.r_[type_start, len(sorted_types)])   # cells per type block (its "number")

def pick_ticks(mids, names, weights, min_gap):
    """Greedily keep the highest-weight labels that stay >= min_gap apart (in cell units)."""
    kept, kept_pos = [], []
    for i in np.argsort(weights)[::-1]:                      # largest first
        c = mids[i]
        if all(abs(c - p) >= min_gap for p in kept_pos):
            kept_pos.append(c)
            kept.append(i)
    kept.sort(key=lambda i: mids[i])                         # back into axis order
    return [int(mids[i]) for i in kept], [str(names[i]) for i in kept]


min_gap = max(1, int(round(LABEL_GAP_PX / px_per_col)))
if ALL_TYPE_LABELS:
    tick_pos = [int(m) for m in type_mid // factor]     # one label per type block, no thinning
    tick_txt = [str(n) for n in type_name]
else:
    tick_pos, tick_txt = pick_ticks(type_mid // factor, type_name, type_size, min_gap)
print(f"showing {len(tick_pos)} / {len(type_name)} type labels"
      + ("" if ALL_TYPE_LABELS else f" (>= {min_gap} cells apart)"))

fig.update_layout(
    title=(f"Per-cell connectivity: neuropil &#8594; cell type "
           f"({'symlog' if USE_SYMLOG else 'linear'} scale, block-max factor {factor})"),
    height=900,
    width=900,
    margin=dict(l=135, b=110, t=95, r=70),
    # same picked ticks on both spines, both rotated 45deg so every label is parallel;
    # tickmode="array" ticks pan/zoom with the axis, so only the ones in view are drawn
    # pin the ranges to the exact data extent -- the dummy scatter that anchors the top axis
    # would otherwise switch these to padded autorange, leaving whitespace at the edges
    xaxis=dict(title="post-synaptic cell", tickmode="array",
               tickvals=tick_pos, ticktext=tick_txt, tickangle=45,
               tickfont=dict(size=9), showticklabels=True,
               range=[-0.5, n_small - 0.5]),
    # ticks="outside" + an invisible tick mark nudges the y labels a little further left
    yaxis=dict(title="pre-synaptic cell", tickmode="array",
               tickvals=tick_pos, ticktext=tick_txt, tickangle=0,
               tickfont=dict(size=9), showticklabels=True,
               ticks="outside", ticklen=10, tickcolor="rgba(0,0,0,0)",
               range=[n_small - 0.5, -0.5]),
    # neuropil names on the top spine, range-locked to x so they scroll with the matrix
    xaxis2=dict(overlaying="x", matches="x", side="top", tickmode="array",
                tickvals=npil_tickpos, ticktext=npil_ticktext, tickangle=0,
                tickfont=dict(size=12), showgrid=False, zeroline=False, showline=False),
)
fig

## 8. Markov chain vs Monte Carlo downstream propagation

A single propagation of the **whole starting set** (L1–L5, R7, R8) 5 hops downstream, shown on
the *exact* grouped cell×cell matrix of Section 5 — only the **values** change per hop:

* **hop 0** — the grouped matrix with every **non-starting row zeroed** (the starting cells and
  their raw output synapses, normalised to its own max for the shared colour scale);
* **hop 1** — the **output contribution** of the starting cells to their direct outputs
  ($\operatorname{diag}(\mathbf{o}_0)\,P_{\text{red}}$, rows = starting cells);
* **hops 2–5** — the output contribution from the **frontier of the previous hop** to *its*
  targets ($\operatorname{diag}(\mathbf{o}_{h-1})\,P_{\text{red}}$), so the bright rows march
  downstream retina → ME → LO → … → targets.

The frontier advances by $\mathbf{o}_{h}=\mathbf{o}_{h-1}P_{\text{red}}$ (the column sums of the
hop-$h$ flow). **Markov** is the exact deterministic flow; **Monte Carlo** estimates it by
counting the $i\!\to\!j$ transitions of many random walkers at each hop. `P_red` is row-stochastic
on the displayed subgraph with LC/LPLC targets (and dead ends) as absorbing self-loops.

To the right, **one jitterplot per input type** shows its **top-10 output-contribution pathways**
(target types) — one marker per source-cell replicate, mean + 95% CI — from the full-matrix `OC`
of Section 3.

In [ ]:
# ==== Section 8 (base cell): shared Markov / Monte-Carlo objects ====
# Recovered from session history (2026-09-02). Run this BEFORE the 8b / 8c / 8d / 8e cells.
# It builds the reduced row-stochastic operator Pr on the grouped Section-5 subgraph and every
# object the downstream panels reuse: the starting distribution o0, the per-hop Markov (markov_R)
# and Monte-Carlo (mc_R) flow matrices, the per-start-type output-contribution table oc_type, and
# the top-pathway table path_by_type.

# ---- parameters -------------------------------------------------------------
N_HOPS = 5
N_WALKERS = 8000                 # MC walkers for the heatmaps (total, over all starting cells)
N_PATH_WALKERS = 6000            # MC walkers per start type for the pathway column
MAT_TARGET = 700                 # block-downsample width for each cell x cell matrix panel
TOP_TARGETS = 10                 # col 3: output-contribution target types per start type
TOP_PATHS = 25                   # col 4: cell-type pathways per start type
CBAR_DECADES = 4                 # colour scale spans this many log10 decades below each panel's peak
SL = 1e-4                        # symlog linear threshold for the output-contribution / probability axes
rng = np.random.default_rng(0)

# ---- reduced, fully row-stochastic operator on the grouped subgraph ---------
Wr = W_sorted.astype(float).tocsr()
outdeg = np.asarray(Wr.sum(1)).ravel()
absorb_r = np.isin(sorted_types, TARGET_TYPES)
selfloop = absorb_r | (outdeg == 0)          # targets + dead ends hold their mass
inv = np.where(outdeg > 0, 1.0 / np.maximum(outdeg, 1.0), 0.0)
Pr = sp.diags(np.where(selfloop, 0.0, inv)) @ Wr
Pr = (Pr + sp.diags(selfloop.astype(float))).tocsr()      # every row sums to 1

start_types = [t for t in STARTING_TYPES if (sorted_types == t).any()]
Nred = Pr.shape[0]
is_start = np.isin(sorted_types, start_types)
o0 = is_start.astype(float) / max(is_start.sum(), 1)      # starting distribution over ALL sources

# ---- Markov + Monte-Carlo frontier-flow matrices ----------------------------
# hop 0 = raw path count of the starting cells; hops 1..N = the frontier's output contribution
# (diag(o_{h-1}) @ Pr, o advancing by o @ Pr).
def markov_flow_matrices():
    mats = [(sp.diags(is_start.astype(float)) @ W_sorted).tocsr()]   # hop 0: raw path count
    o = o0.copy()
    for _ in range(1, N_HOPS + 1):
        mats.append((sp.diags(o) @ Pr).tocsr())            # frontier -> its targets
        o = np.asarray(o @ Pr).ravel()                     # advance the frontier
    return mats

def sample_next(pos):
    rows = Pr[pos]
    indptr, indices, data = rows.indptr, rows.indices, rows.data
    r = rng.random(len(pos))
    nxt = np.empty(len(pos), dtype=np.int64)
    for k in range(len(pos)):
        a, b = indptr[k], indptr[k + 1]
        c = np.cumsum(data[a:b])
        nxt[k] = indices[a + np.searchsorted(c, r[k] * c[-1])]
    return nxt

def mc_flow_matrices():
    mats = [(sp.diags(is_start.astype(float)) @ W_sorted).tocsr()]   # hop 0: same raw path count
    pos = rng.choice(np.flatnonzero(is_start), size=N_WALKERS)       # walkers uniform over sources
    for _ in range(1, N_HOPS + 1):
        nxt = sample_next(pos)
        mats.append(sp.coo_matrix((np.full(len(pos), 1.0 / N_WALKERS), (pos, nxt)),
                                  shape=(Nred, Nred)).tocsr())        # i->j transition flow
        pos = nxt
    return mats

markov, mc = markov_flow_matrices(), mc_flow_matrices()

# ---- block-downsample a cell x cell matrix (mass-preserving sum) ------------
factor = max(1, int(np.ceil(Nred / MAT_TARGET)))
nout = int(np.ceil(Nred / factor))

def reduce_mat(M):
    coo = M.tocoo()
    if coo.nnz == 0:
        return np.zeros((nout, nout))
    lin = (coo.row // factor) * nout + (coo.col // factor)
    order = np.argsort(lin, kind="stable")
    lin_s, data_s = lin[order], coo.data[order]
    uniq, starts = np.unique(lin_s, return_index=True)
    out = np.zeros(nout * nout)
    out[uniq] = np.add.reduceat(data_s, starts)
    return out.reshape(nout, nout)

markov_R = [reduce_mat(m) for m in markov]
mc_R = [reduce_mat(m) for m in mc]

# ---- col 3: top output-contribution target types per start type -------------
src_ct = node_info.cell_type.astype(str).values[src_idx]
tgt_ct = node_info.cell_type.astype(str).values[tgt_idx]
tgt_types, tgt_codes = np.unique(tgt_ct, return_inverse=True)
G_tgt = sp.csr_matrix((np.ones(len(tgt_codes)), (np.arange(len(tgt_codes)), tgt_codes)),
                      shape=(len(tgt_codes), len(tgt_types)))
oc_type = np.asarray((sub @ G_tgt)) / np.maximum(rowsum[:, None], 1e-12)

# ---- col 4: top Monte-Carlo pathways (cell-type sequences) per start type ----
# a pathway == the walker's per-hop cell-type sequence up to and INCLUDING the first LC/LPLC
# target it reaches. Only LC/LPLC targets are stopping points -- start types and generic dead ends
# are NOT, so they can't appear as pathway endpoints. Walkers that never reach a target are dropped.
def sample_pathways(start_cells, n):
    pos = rng.choice(start_cells, size=n)
    seqs = [sorted_types[pos].copy()]
    hit = [absorb_r[pos].copy()]                 # LC/LPLC target reached? (targets only)
    done = absorb_r[pos].copy()
    for _ in range(N_HOPS):
        active = ~done
        nxt = pos.copy()
        if active.any():
            nxt[active] = sample_next(pos[active])
        pos = nxt
        seqs.append(sorted_types[pos].copy())
        hit.append(absorb_r[pos].copy())
        done |= absorb_r[pos]
    seqs = np.array(seqs)
    hit = np.array(hit)
    from collections import Counter
    counts = Counter()
    for w in range(n):
        f = hit[:, w]
        if not f.any():
            continue                             # never reached a target -> no target pathway
        cut = int(np.argmax(f)) + 1              # truncate at (and include) the first target
        counts[" > ".join(seqs[:cut, w])] += 1
    labels, cnt = zip(*counts.most_common(TOP_PATHS)) if counts else ([], [])
    p = np.array(cnt, dtype=float) / n           # fraction of ALL walkers taking that pathway
    return list(labels), p, np.sqrt(p * (1 - p) / n)

path_by_type = {st: sample_pathways(np.flatnonzero(sorted_types == st), N_PATH_WALKERS)
                for st in start_types}

print(f"Section 8 base ready: Nred={Nred}, reduced {nout}x{nout} (factor {factor}), "
      f"{len(markov_R)} hops")
print("start_types:", start_types)

In [ ]:
# ==== Section 8c: Markov-only panel (no Monte Carlo), full Markov per-hop colour range ====
# Run AFTER the Section 8 cell.
from plotly.subplots import make_subplots

GRID = "lightgray"          # gray gridlines on a white background for the scatter columns
N_HOPS = 5
NCOLS3, NROWS3 = 3, max(N_HOPS + 1, len(start_types))
CW3 = [0.28, 0.36, 0.36]
HS3, VS3, SQ3 = 0.16, 0.03, 230
L3, R3, T3, B3 = 90, 90, 100, 60
sx3, sy3 = 1 - HS3 * (NCOLS3 - 1), 1 - VS3 * (NROWS3 - 1)
wn3 = [w / sum(CW3) * sx3 for w in CW3]
W3, H3 = int(SQ3 / wn3[0] + L3 + R3), int(SQ3 * NROWS3 / sy3 + T3 + B3)
domx3, x = [], 0.0
for w in wn3:
    domx3.append((x, x + w)); x += w + HS3
rh3 = sy3 / NROWS3
def rc3(r):
    return 1 - (r - 1) * (rh3 + VS3) - rh3 / 2
CBX3 = domx3[0][1] + 0.006

titles3 = []
for row in range(NROWS3):
    titles3 += (["Markov chain", "", ""] if row == 0 else ["", "", ""])
fig = make_subplots(rows=NROWS3, cols=NCOLS3, column_widths=CW3, horizontal_spacing=HS3,
                    vertical_spacing=VS3, subplot_titles=titles3)

for h in range(N_HOPS + 1):
    pos = markov_R[h][markov_R[h] > 0]
    # vmin = float(pos.min()) if pos.size else 1e-12
    # vmax = float(markov_R[h].max()) if pos.size else 1.0
    vmin, vmax = 10e-8, 10e-4
    zmin, zmax = np.log10(vmin), np.log10(max(vmax, vmin * 10))
    # base-10 colorbar: one tick per decade, labelled 10^n
    decs = list(range(int(np.floor(zmin)), int(np.ceil(zmax)) + 1))
    if len(decs) > 7:
        decs = decs[:: int(np.ceil(len(decs) / 7))]
    fig.add_trace(go.Heatmap(z=np.log10(markov_R[h] + 1e-30), zmin=zmin, zmax=zmax,
                             colorscale="viridis", showscale=True,
                             colorbar=dict(x=CBX3, y=rc3(h + 1), len=rh3 * 0.9, thickness=9,
                                           tickfont=dict(size=12), tickvals=decs,
                                           ticktext=[f"10<sup>{e}</sup>" for e in decs],
                                           title=dict(text=f"hop {h}", side="right", font=dict(size=9)))),
                  row=h + 1, col=1)
    fig.update_yaxes(range=[nout - 0.5, -0.5], showticklabels=False, row=h + 1, col=1)
    fig.update_xaxes(range=[-0.5, nout - 0.5], showticklabels=False, row=h + 1, col=1)
    fig.add_annotation(x=-0.012, xref="paper", y=rc3(h + 1), yref="paper", text=f"hop {h}",
                       showarrow=False, xanchor="right", yanchor="middle", font=dict(size=12))

bounds = (npil_start[1:] // factor) - 0.5
for b in bounds:
    fig.add_vline(x=b, line=dict(color="rgba(255,255,255,0.5)", width=0.5), row="all", col=1)
    fig.add_hline(y=b, line=dict(color="rgba(255,255,255,0.5)", width=0.5), row="all", col=1)
fig.update_xaxes(tickmode="array", tickvals=list(npil_mid // factor), ticktext=list(npil_name),
                 tickangle=45, tickfont=dict(size=7), showticklabels=True, row=N_HOPS + 1, col=1)

def symlog_axis3(row, col, title):
    raw = [0.0, SL, 1e-3, 1e-2, 1e-1, 1.0]
    fig.update_xaxes(range=[0, float(symlog(1.0, SL)) + 0.3], tickmode="array",
                     tickvals=[float(symlog(v, SL)) for v in raw],
                     ticktext=["0", "10\u207b\u2074", "10\u207b\u00b3", "10\u207b\u00b2", "10\u207b\u00b9", "1"],
                     tickfont=dict(size=7), showticklabels=(row == NROWS3),
                     showgrid=True, gridcolor=GRID, zeroline=False,
                     title_text=title if row == NROWS3 else "", row=row, col=col)

for i, st in enumerate(start_types):
    rows_s = np.flatnonzero(src_ct == st)
    top = np.argsort(oc_type[rows_s].mean(0))[::-1][:TOP_TARGETS]
    for rank, col_t in enumerate(top):
        vals = oc_type[rows_s, col_t]; y0 = TOP_TARGETS - 1 - rank
        m = vals.mean(); sem = vals.std(ddof=1) / np.sqrt(max(len(vals), 1)) if len(vals) > 1 else 0.0
        fig.add_trace(go.Scatter(x=symlog(vals, SL), y=y0 + np.random.normal(0, 0.06, len(vals)),
                                 mode="markers", marker=dict(size=3, color="rgba(128,128,128,0.5)"),
                                 hovertext=[tgt_types[col_t]] * len(vals), showlegend=False), row=i + 1, col=2)
        fig.add_trace(go.Scatter(x=symlog(np.maximum([m - 1.96 * sem, m + 1.96 * sem], 0), SL),
                                 y=[y0, y0], mode="lines", line=dict(color="black", width=1.5),
                                 showlegend=False), row=i + 1, col=2)
        fig.add_trace(go.Scatter(x=symlog([m], SL), y=[y0], mode="markers",
                                 marker=dict(size=7, color="black", symbol="diamond",
                                             line=dict(color="white", width=1)),
                                 showlegend=False), row=i + 1, col=2)
    fig.update_yaxes(tickmode="array", tickvals=list(range(TOP_TARGETS)),
                     ticktext=[tgt_types[c] for c in top[::-1]], tickfont=dict(size=14),
                     showgrid=True, gridcolor=GRID,
                     title_text=st, title_font=dict(size=12), row=i + 1, col=2)
    symlog_axis3(i + 1, 2, "output contribution")

for i, st in enumerate(start_types):
    labels, p, sem = path_by_type[st]; npath = len(labels)
    for rank in range(npath):
        y0 = TOP_PATHS - 1 - rank
        lo, hi = max(p[rank] - 1.96 * sem[rank], 0), p[rank] + 1.96 * sem[rank]
        fig.add_trace(go.Scatter(x=symlog(np.array([lo, hi]), SL), y=[y0, y0], mode="lines",
                                 line=dict(color="rgba(90,90,90,0.7)", width=1.2), showlegend=False),
                      row=i + 1, col=3)
        fig.add_trace(go.Scatter(x=symlog([p[rank]], SL), y=[y0], mode="markers",
                                 marker=dict(size=5, color="black"),
                                 hovertext=[labels[rank]], showlegend=False), row=i + 1, col=3)
    fig.update_yaxes(tickmode="array", tickvals=list(range(TOP_PATHS)),
                     ticktext=[labels[TOP_PATHS - 1 - r] if (TOP_PATHS - 1 - r) < npath else ""
                               for r in range(TOP_PATHS)],
                     tickfont=dict(size=10), range=[-0.5, TOP_PATHS - 0.5],
                     showgrid=True, gridcolor=GRID, row=i + 1, col=3)
    symlog_axis3(i + 1, 3, "pathway probability")

fig.update_layout(height=H3, width=W3, margin=dict(l=L3, r=R3, t=T3, b=B3), plot_bgcolor="white",
                  title=("Markov-only downstream propagation (full per-hop colour range) · "
                         f"top-{TOP_TARGETS} target output contributions · top-{TOP_PATHS} pathways"))
# output to an html file
fig.write_html("markov_only_downstream_propagation.html")
# open using wbbrowser
import webbrowser
webbrowser.open("markov_only_downstream_propagation.html")

In [ ]:
# ==== Section 8b: high-detail Markov vs Monte-Carlo heatmaps (two colour-scaling versions) ====
# Run AFTER the Section 8 cell (needs markov_R, mc_R, nout, factor, npil_*, N_HOPS, N_WALKERS).
from plotly.subplots import make_subplots

def heatmap_columns(scale_by="markov", cell_px=380):
    """Large 2-column (Markov | Monte Carlo) hop-by-hop heatmaps.

    scale_by : "markov" or "mc" -- per hop, vmin (smallest nonzero) / vmax are taken from that
               source's own panel and applied to BOTH columns so they're directly comparable.
    """
    src = markov_R if scale_by == "markov" else mc_R
    nrows = N_HOPS + 1
    vs, L, R, T, B = 0.012, 80, 120, 70, 60
    sy = 1 - vs * (nrows - 1)
    rh = sy / nrows
    height = int(cell_px * nrows / sy + T + B)
    width = int(cell_px / 0.48 + L + R)                    # two square cols (0.48 each) + gutter
    def row_center(r):
        return 1 - (r - 1) * (rh + vs) - rh / 2

    titles = ["Markov chain", "Monte Carlo"] + [""] * (2 * nrows - 2)
    fig = make_subplots(rows=nrows, cols=2, horizontal_spacing=0.04, vertical_spacing=vs,
                        subplot_titles=titles)
    for h in range(nrows):
        pos = src[h][src[h] > 0]
        # vmin = float(pos.min()) if pos.size else 1.0 / N_WALKERS
        # vmax = float(src[h].max()) if pos.size else 1.0
        vmin = float(np.percentile(pos, 5)) if pos.size else 1.0 / N_WALKERS
        vmax = float(np.percentile(pos, 95)) if pos.size else 1.0
        zmin, zmax = np.log10(vmin), np.log10(max(vmax, vmin * 10))
        tv = np.linspace(zmin, zmax, 6)
        for col, R_ in ((1, markov_R[h]), (2, mc_R[h])):
            fig.add_trace(go.Heatmap(z=np.log10(R_ + 1e-30), zmin=zmin, zmax=zmax,
                                     colorscale="magma", showscale=(col == 2),
                                     colorbar=dict(x=1.005, y=row_center(h + 1), len=rh * 0.9,
                                                   thickness=11, tickfont=dict(size=8),
                                                   tickvals=list(tv),
                                                   ticktext=[f"{10 ** t:.1e}" for t in tv],
                                                   title=dict(text=f"hop {h}", side="right",
                                                              font=dict(size=9)))),
                          row=h + 1, col=col)
            fig.update_yaxes(range=[nout - 0.5, -0.5], showticklabels=False, row=h + 1, col=col)
            fig.update_xaxes(range=[-0.5, nout - 0.5], showticklabels=False, row=h + 1, col=col)
        fig.add_annotation(x=-0.011, xref="paper", y=row_center(h + 1), yref="paper",
                           text=f"hop {h}", showarrow=False, xanchor="right", yanchor="middle",
                           font=dict(size=13))
    bounds = (npil_start[1:] // factor) - 0.5
    for col in (1, 2):
        for b in bounds:
            fig.add_vline(x=b, line=dict(color="rgba(255,255,255,0.5)", width=0.5), row="all", col=col)
            fig.add_hline(y=b, line=dict(color="rgba(255,255,255,0.5)", width=0.5), row="all", col=col)
        fig.update_xaxes(tickmode="array", tickvals=list(npil_mid // factor), ticktext=list(npil_name),
                         tickangle=45, tickfont=dict(size=8), showticklabels=True, row=N_HOPS + 1, col=col)
    fig.update_layout(height=height, width=width, margin=dict(l=L, r=R, t=T, b=B),
                      title=f"Markov vs Monte Carlo heatmaps — colour range from {scale_by.upper()} min/max per hop")
    return fig

heatmap_columns("markov").show()   # version scaled to the Markov min/max per hop
heatmap_columns("mc").show()       # version scaled to the Monte Carlo min/max per hop

In [ ]:
# ==== Section 8d: 3D meshes per hop, opacity binned into levels by output contribution ====
# Run AFTER the Section 8 cell (needs o0, Pr, N_HOPS, sorted_info, markov_R, nout, factor, npil_*).
import pickle
import navis
from fafbseg import flywire
from plotly.subplots import make_subplots

# ---- parameters -------------------------------------------------------------
MESH_LOD    = 6          # FlyWire mesh level-of-detail (higher = coarser/faster; try 3-5)
N_LEVELS    = 5          # opacity buckets
MIN_OP, MAX_OP = 0.04, 0.5   # opacity of the lowest / highest level
VAL_FLOOR   = 1e-9       # cells with occupancy below this are not drawn at any hop
MESH_COLOR  = "black"

root_ids = sorted_info.root_id.values.astype(np.int64)
# ---- Select cells: either first N or top N by occupancy ----
SELECT_BY_OC = True      # True = top N by total occupancy; False = first N
MAX_CELLS   = 4000      # None = full reduced set; a number = quick test on the first/top N cells

root_ids_all = sorted_info.root_id.values.astype(np.int64)

if MAX_CELLS is not None:
    if SELECT_BY_OC:
        # Per-cell occupancy: sum of log(OC) across all hops (to weight higher hops more)
        o0_full = o0.copy()
        o = o0_full.copy()
        per_cell_oc = np.log10(np.maximum(o0_full, VAL_FLOOR))  # hop 0
        for _ in range(1, N_HOPS + 1):
            o = np.asarray(o @ Pr).ravel()
            per_cell_oc += np.log10(np.maximum(o, VAL_FLOOR))
        
        # Sort by occupancy, take top MAX_CELLS
        top_idx = np.argsort(per_cell_oc)[::-1][:MAX_CELLS]
        root_ids = root_ids_all[top_idx]
        print(f"Selected top {len(root_ids)} cells by occupancy")
    else:
        # Just take the first MAX_CELLS
        root_ids = root_ids_all[:MAX_CELLS]
        print(f"Selected first {len(root_ids)} cells")
else:
    root_ids = root_ids_all
    print(f"Using all {len(root_ids)} cells")

# ---- per-hop node occupancy o_h (output contribution present at each cell) ---
node_vals = [o0.copy()]
o = o0.copy()
for _ in range(1, N_HOPS + 1):
    o = np.asarray(o @ Pr).ravel()
    node_vals.append(o.copy())
node_vals = [v[:len(root_ids)] for v in node_vals]     # align to the (possibly capped) cell set

# ---- fetch + cache meshes (vertices/faces per root_id) -----------------------
MESH_PKL = CACHE_DIR / f"meshes_lod{MESH_LOD}.pkl"
meshes = pickle.load(open(MESH_PKL, "rb")) if MESH_PKL.exists() else {}
missing = [int(r) for r in root_ids if int(r) not in meshes]
if missing:
    print(f"fetching {len(missing)} meshes (lod {MESH_LOD}) -- slow the first time...")
    for c0 in range(0, len(missing), 50):                # chunk to keep requests sane
        chunk = missing[c0:c0 + 50]
        try:
            for m in navis.NeuronList(flywire.get_mesh_neuron(chunk, lod=MESH_LOD)):
                meshes[int(m.id)] = (np.asarray(m.vertices, np.float32),
                                     np.asarray(m.faces, np.int64))
        except Exception as e:
            print(f"  chunk {c0}-{c0+len(chunk)} failed: {e}")
        print(f"  {min(c0 + 50, len(missing))}/{len(missing)}")
    pickle.dump(meshes, open(MESH_PKL, "wb"))
have = np.array([int(r) in meshes for r in root_ids])
print(f"meshes available for {have.sum()}/{len(root_ids)} cells")

# ---- global log-spaced level edges (shared across hops so opacity is comparable) ----
allv = np.concatenate([v[v > VAL_FLOOR] for v in node_vals])
lo, hi = float(allv.min()), float(allv.max())
edges = np.logspace(np.log10(lo), np.log10(hi), N_LEVELS + 1)
level_op = np.linspace(MIN_OP, MAX_OP, N_LEVELS)         # opacity per level (low OC -> high OC)

# def combined_mesh(idx):
#     """Concatenate the meshes of cell indices `idx` into one (V, F) with offset faces."""
#     Vs, Fs, off = [], [], 0
#     for i in idx:
#         rid = int(root_ids[i])
#         if rid not in meshes:
#             continue
#         V, F = meshes[rid]
#         Vs.append(V); Fs.append(F + off); off += len(V)
#     if not Vs:
#         return None
#     return np.vstack(Vs), np.vstack(Fs)

# # ---- figure: per hop, heatmap (col 1) + 3D mesh (col 2) ----------------------
# NROWS = N_HOPS + 1
# fig = make_subplots(rows=NROWS, cols=2, column_widths=[0.5, 0.5],
#                     horizontal_spacing=0.03, vertical_spacing=0.02,
#                     specs=[[{"type": "xy"}, {"type": "scene"}] for _ in range(NROWS)],
#                     subplot_titles=["Markov chain", "cells (opacity ~ log OC)"] + [""] * (2 * NROWS - 2))

# for h in range(NROWS):
#     # heatmap
#     R = markov_R[h]
#     pos = R[R > 0]
#     zmin = np.log10(pos.min()) if pos.size else -6
#     zmax = np.log10(R.max()) if pos.size else 0
#     fig.add_trace(go.Heatmap(z=np.log10(R + 1e-30), zmin=zmin, zmax=zmax,
#                              colorscale="magma", showscale=False), row=h + 1, col=1)
#     fig.update_yaxes(range=[nout - 0.5, -0.5], showticklabels=False, row=h + 1, col=1)
#     fig.update_xaxes(range=[-0.5, nout - 0.5], showticklabels=False, row=h + 1, col=1)

#     # 3D meshes, one trace per opacity level
#     vals = node_vals[h]
#     lvl = np.clip(np.digitize(vals, edges) - 1, 0, N_LEVELS - 1)
#     lvl[vals <= VAL_FLOOR] = -1                          # skip near-zero cells
#     for b in range(N_LEVELS):
#         idx = np.flatnonzero(lvl == b)
#         cm = combined_mesh(idx)
#         if cm is None:
#             continue
#         V, F = cm
#         fig.add_trace(go.Mesh3d(x=V[:, 0], y=V[:, 1], z=V[:, 2],
#                                 i=F[:, 0], j=F[:, 1], k=F[:, 2],
#                                 color=MESH_COLOR, opacity=float(level_op[b]),
#                                 flatshading=True, lighting=dict(ambient=0.9, diffuse=0.1),
#                                 hoverinfo="skip", showscale=False), row=h + 1, col=2)
#     sc = "scene" if h == 0 else f"scene{h + 1}"
#     fig.layout[sc].update(aspectmode="data",
#                           xaxis=dict(visible=False), yaxis=dict(visible=False), zaxis=dict(visible=False))

# fig.update_layout(height=380 * NROWS, width=1100, margin=dict(l=40, r=20, t=60, b=20),
#                   title=f"Per-hop output contribution on the reduced-set meshes (lod {MESH_LOD}, {N_LEVELS} opacity levels)")
# # output to html and open
# fig.to_html("markov_meshes_per_hop.html")
# import webbrowser
# webbrowser.open("markov_meshes_per_hop.html") 

In [ ]:
# ==== Section 8e: one neuroglancer scene per hop, cells coloured by magma (log OC), low layer alpha ====
# Run AFTER the Section 8 cell (needs o0, Pr, N_HOPS, sorted_info). No mesh fetching required.
import json
from urllib.parse import unquote, quote
import matplotlib
import matplotlib.pyplot as plt
import matplotlib.colors as mcolors
from fafbseg import flywire
from IPython.display import HTML, display

# ---- parameters -------------------------------------------------------------
LAYER_ALPHA = 0.15        # global opacity of the whole segmentation layer (3D meshes)
VAL_FLOOR   = 1e-9        # cells with occupancy below this are omitted from the scene
# load the magma colormap
MAGMA       = matplotlib.colormaps["magma"]

root_ids = sorted_info.root_id.values.astype(np.int64)

# ---- per-hop node occupancy o_h (output contribution present at each cell) ---
node_vals = [o0.copy()]
o = o0.copy()
for _ in range(1, N_HOPS + 1):
    o = np.asarray(o @ Pr).ravel()
    node_vals.append(o.copy())

def magma_hex(norm01):
    return mcolors.to_hex(MAGMA(float(norm01)))

def set_layer_alpha(url, alpha):
    """Patch every segmentation layer's mesh/2D alpha in a neuroglancer URL state."""
    if "#!" not in url:
        return url
    base, frag = url.split("#!", 1)
    state = json.loads(unquote(frag))
    for lyr in state.get("layers", []):
        if lyr.get("type") == "segmentation":
            lyr["objectAlpha"] = alpha          # 3D mesh opacity
            lyr["selectedAlpha"] = alpha        # 2D cross-section (selected)
            lyr["notSelectedAlpha"] = alpha     # 2D cross-section (rest)
    return base + "#!" + quote(json.dumps(state))

# ---- build one scene per hop ------------------------------------------------
hop_urls = []
for h in range(N_HOPS + 1):
    v = node_vals[h]
    pos = v > VAL_FLOOR
    lz = np.log10(np.where(pos, v, VAL_FLOOR))
    zmin, zmax = float(lz[pos].min()), float(lz[pos].max())   # same log scaling idea as the heatmaps
    norm = np.clip((lz - zmin) / (zmax - zmin + 1e-12), 0, 1)
    seg_colors = {int(root_ids[i]): magma_hex(norm[i]) for i in np.flatnonzero(pos)}

    url = flywire.encode_url(segments=list(seg_colors.keys()),
                             seg_colors=seg_colors, open=False)
    url = set_layer_alpha(url, LAYER_ALPHA)
    hop_urls.append(url)
    print(f"hop {h}: {len(seg_colors)} cells")

# clickable links in the notebook
display(HTML("<br>".join(f'<a href="{u}" target="_blank">hop {h} — neuroglancer</a>'
                         for h, u in enumerate(hop_urls))))

### The remaining (non-optic) neuropils

The cells above are restricted to the optic lobe and LC-target neuropils of one hemisphere.
The rest sit in a long tail of central-brain neuropils that mostly connect **within
themselves**. Rather than clutter the per-cell matrix with them, we summarise them at the
neuropil level (keeping the `_R` / `_L` side, so "self" means the same neuropil on the same
side): a neuropil x neuropil matrix of mean synapses per cell pair, plus each neuropil's
within-neuropil (self) connection fraction.

In [ ]:
from IPython.display import display

# non-optic cells (everything not in MAIN_NEUROPILS), summarised at the neuropil level
other_keep = ~npil_base.isin(MAIN_NEUROPILS).values
oinfo = node_info[other_keep].copy()
W_other = W[other_keep][:, other_keep].tocsr()

Go, onp_names, onp_counts = type_indicator(oinfo, type_col="neuropil")
tot = np.asarray((Go.T @ W_other @ Go).todense())         # total synapses between neuropils
mean_block = tot / np.outer(onp_counts, onp_counts)        # mean per cell pair

# order by total connectivity so the busiest neuropils lead
ordr = np.argsort(tot.sum(0) + tot.sum(1))[::-1]
onp_names, onp_counts = onp_names[ordr], onp_counts[ordr]
tot, mean_block = tot[np.ix_(ordr, ordr)], mean_block[np.ix_(ordr, ordr)]

# fraction of each neuropil's synapses (within the non-optic subset) that stay within itself
self_frac = np.diag(tot) / np.maximum(tot.sum(1), 1)
print(f"{other_keep.sum():,} cells across {len(onp_names)} non-optic neuropils")
display(pd.DataFrame({"neuropil": onp_names, "n_cells": onp_counts,
                      "self_conn_frac": self_frac.round(3)}).head(30))

# neuropil x neuropil heatmap (top 40 by connectivity): a bright diagonal = self-connection
top = slice(0, 40)
heatmap(mean_block[top, top], list(onp_names[top]),
        "Non-optic neuropils: mean synapses per cell pair", height=800)

## 6. Worked example matching the cartoon

Pick one source cell and one target cell and reproduce the three numbers
annotated on `network_measures_cartoon.svg`.

In [ ]:
def report(source_root_id, target_root_id):
    s_pos = int(np.flatnonzero(node_info.root_id.values == source_root_id)[0])
    t_pos = int(np.flatnonzero(node_info.root_id.values == target_root_id)[0])
    s_row = int(np.flatnonzero(src_idx == s_pos)[0])
    t_col = int(np.flatnonzero(tgt_idx == t_pos)[0])

    pc = sub[s_row, t_col]
    print(f"source {source_root_id} ({node_info.cell_type.iloc[s_pos]})")
    print(f"target {target_root_id} ({node_info.cell_type.iloc[t_pos]})")
    print(f"  path count          W(s->t) = {pc:,.0f}")
    print(f"  input  contribution IC      = {IC[s_row, t_col]:.4f}")
    print(f"  output contribution OC      = {OC[s_row, t_col]:.4f}")


# strongest source-target pair in the subgraph
best = np.unravel_index(np.argmax(sub), sub.shape)
report(node_info.root_id.iloc[src_idx[best[0]]], node_info.root_id.iloc[tgt_idx[best[1]]])

## 7. Hop by hop: L1 → LC11

Taking **L1** as the source and **LC11** as the target, we can watch mass move along the
canonical **L1 → Mi1 → T3 → LC11** pathway by propagating one hop at a time instead of
summing over hops. Starting one unit on every L1 cell and aggregating the frontier by cell
type after each hop, the mass peaks in **Mi1** at hop 1, **T3** at hop 2, and first reaches
the **LC11** target at hop 3 -- the three legs of the pathway show up as successive peaks.
Swapping `W_eff` for `P_eff` turns the per-hop synapse-weighted path counts into
first-passage probabilities.

In [ ]:
import matplotlib.pyplot as plt

G, type_names, _ = type_indicator(node_info)
type_pos = {t: i for i, t in enumerate(type_names)}


def per_hop_by_type(M, source_type="L1", n_hops=MAX_HOPS):
    """Total mass reaching each cell type at each hop, starting from `source_type`.

    One unit of mass starts on every `source_type` cell and is propagated one hop at a time
    through `M` (use W_eff for synapse-weighted path counts, P_eff for first-passage
    probabilities). `M` is absorbing at the LC/LPLC targets, so mass that reaches a target is
    not relayed onward. Returns a (n_hops x n_types) DataFrame.
    """
    v = node_info.cell_type.astype(str).eq(source_type).values.astype(float)
    rows = []
    for _ in range(n_hops):
        v = M.T @ v                         # frontier one hop further along
        rows.append(np.asarray(G.T @ v).ravel())
    return pd.DataFrame(rows, index=[f"hop {h}" for h in range(1, n_hops + 1)],
                        columns=type_names)


counts = per_hop_by_type(W_eff)             # synapse-weighted path counts
probs = per_hop_by_type(P_eff)              # first-passage probabilities

track = [t for t in ["Mi1", "T3", "LC11"] if t in type_pos]
missing = [t for t in ["Mi1", "T3", "LC11"] if t not in type_pos]
if missing:
    print("not present as exact type names in this subgraph:", missing)
print("synapse-weighted path counts from L1, by hop:")
print(counts[track].round(0), "\n")
print("first-passage probability mass from L1, by hop:")
print(probs[track].round(4))

fig, ax = plt.subplots(figsize=(6, 3.5))
for t in track:
    ax.plot(range(1, MAX_HOPS + 1), counts[t].values, "o-", label=t)
ax.set_yscale("log")
ax.set_xticks(range(1, MAX_HOPS + 1))
ax.set_xlabel("hop")
ax.set_ylabel("path count from L1  (synapse-weighted)")
ax.set_title("L1 \u2192 Mi1 \u2192 T3 \u2192 LC11, one hop at a time")
ax.legend()
plt.show()